In [1]:
print('-------importing modules-----------------')
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
import os
from loader import load_pkl, load_checkpoint
from model import NMF
from preprocessor import preprocessor
from dataset_loader import Test_Rating_dataSet
from itertools import islice
import operator
import ast


-------importing modules-----------------


In [2]:

def str_to_list(list_string):
  return ast.literal_eval(list_string)



In [3]:

print('-------importing modules done-----------------')
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda")
print("Using: ",device)

load_model = True
save_model = False
load_pkl_flag = True
data_path = 'train.csv'
keyword_len = 20

print('----------Preprocessing(loading data)--------------------')
if os.path.isfile('../data/pre.pkl'):
    pre = load_pkl('../data/pre.pkl')
    print('Successfully Loaded Pickle file')
elif os.path.isfile('../data/train.csv'):
    print('No Pickle file found.')
    print('Using CSV file, train.csv.')
    data_path = '../data/train.csv'
    print('Starting Pre-processing...')
    pre = preprocessor(data_path,keywords_limit = keyword_len)
    if pre:
        print('Successfully Loaded CSV file to train')
print('----------Preprocessing Completed--------------------')

-------importing modules done-----------------
Using:  cuda
----------Preprocessing(loading data)--------------------
No Pickle file found.
Using CSV file, train.csv.
Starting Pre-processing...
Successfully Loaded CSV file to train
----------Preprocessing Completed--------------------


In [4]:
dropout = 0.01
learning_rate = 0.001
print("finish loading Data...)")

print("Initailizing Model...")
model_name = '../trained_models/NCF_checkpoint_cuda.pth.tar'
nlayer = 3
dropout = 0.001
model = NMF(user_emb_sizes = pre.user_emb_sizes, url_emb_sizes = pre.url_emb_sizes,url_to_keyword_dict= pre.url_to_keyword_dict ,  nlayer = nlayer, dropout = dropout, device = device).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr= learning_rate)

if load_model:
    print("loading Model.....")
    load_checkpoint(torch.load(model_name), model, optimizer)
    model.train() 
    print("Finish Loading Model")
 

finish loading Data...)
Initailizing Model...
self.MLP_layers  Sequential(
  (0): Linear(in_features=9216, out_features=4608, bias=True)
  (1): Dropout(p=0.001, inplace=False)
  (2): Linear(in_features=4608, out_features=2304, bias=True)
  (3): ReLU()
  (4): Dropout(p=0.001, inplace=False)
  (5): Linear(in_features=2304, out_features=1152, bias=True)
  (6): ReLU()
  (7): Dropout(p=0.001, inplace=False)
  (8): Linear(in_features=1152, out_features=576, bias=True)
  (9): ReLU()
  (10): Dropout(p=0.001, inplace=False)
)
loading Model.....
=> Loading checkpoint
=> Done Loading checkpoint
Finish Loading Model


In [6]:
url_list =  [pre.index_url_mapping[i] for i in range(255)]
url_list

['https://www.corporatenepal.com/news/detail/201726',
 'https://www.corporatenepal.com/news/detail/201739',
 'https://www.corporatenepal.com/news/detail/201741',
 'https://www.corporatenepal.com/news/detail/201744',
 'https://www.corporatenepal.com/news/detail/201749',
 'https://www.corporatenepal.com/news/detail/201750',
 'https://www.corporatenepal.com/news/detail/201752',
 'https://www.corporatenepal.com/news/detail/201755',
 'https://www.corporatenepal.com/news/detail/201765',
 'https://www.corporatenepal.com/news/detail/201770',
 'https://www.corporatenepal.com/news/detail/201784',
 'https://www.corporatenepal.com/news/detail/201786',
 'https://www.corporatenepal.com/news/detail/201788',
 'https://www.corporatenepal.com/news/detail/201791',
 'https://www.corporatenepal.com/news/detail/201792',
 'https://www.corporatenepal.com/news/detail/201794',
 'https://www.corporatenepal.com/news/detail/201795',
 'https://www.corporatenepal.com/news/detail/201797',
 'https://www.corporatenepal

In [12]:
len(pre.index_url_mapping.values())

13856

In [ ]:
df = pd.read_csv("")

In [68]:

def eval_sample_list(user_list,url_list):
  user_df = pd.DataFrame({'user': user_list})
  article_df = pd.DataFrame({'article':url_list})
  test_ds = Test_Rating_dataSet(user_df,article_df, pre.user_index_mapping, pre.url_to_index_and_labels_dict)
  test_dl = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False,)
  url_to_rating = {}
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  for _, (user_name,user,article_url,url_index,labels,Dominant_Topic) in tqdm(enumerate(test_dl)):
      user = user[0].unsqueeze(1)
      # print('keywords before: ',keywords)
      # keywords = torch.tensor([ str_to_list(key) for key in keywords])
      # print('keywords after: ',keywords,keywords.shape)
      user,url_index, label,Dominant_Topic = user.to(device),url_index.to(device),labels.to(device),Dominant_Topic.to(device)
      # print('user shape: ',user.shape,'keywords shape:',keywords.shape,'label shape:',label.shape)
      prediction, user_embed_MLP, article_embed_MLP = model( user, url_index,label,Dominant_Topic)
        
      for i,url in enumerate(article_url[0]):
          
          url_r= {url : float(prediction[i])}
          url_to_rating.update(url_r)

  return url_to_rating


def take(n, iterable):
    "Return first n items of the iterable as a list"
    return list(islice(iterable, n))

    
def get_top_20(user_name,url_list, model):
  user_list = [user_name,]
  url_to_rating = eval_sample_list(user_list,url_list)
  sorted_top_url = dict( sorted(url_to_rating.items(), key=operator.itemgetter(1),reverse=True))
  top_20 = take(20, sorted_top_url.items())
  return top_20

In [67]:
pre.df[pre.df.url == 'https://www.corporatenepal.com/news/detail/201726'].iloc[0]

user                               a769165804e9fef4df0dbacf8eb41408
url               https://www.corporatenepal.com/news/detail/201726
date                                      2077-06-11 00:00:00+05:45
keywords          [66385, 29106, 41200, 40113, 6344, 18899, 4782...
label                                                    अर्थतन्त्र
Dominant_Topic                                                    7
content_list      ['काठमाडौँ', 'उपत्यका', 'व्यापारिक', 'केन्द्र'...
all_content       काठमाडौँ उपत्यका व्यापारिक केन्द्र प्रतीक्षा’ ...
main_topic                                                     अर्थ
final_rating                                               0.516429
user_index                                                    66183
url_index                                                         0
label_index                                                      10
Name: 3383, dtype: object

In [77]:
pre.df[pre.df.url == 'https://www.corporatenepal.com/news/detail/201726'].iloc[0].date

'2077-06-11 00:00:00+05:45'

In [69]:
user_name = '02d8b818b3ece264239f15ad4fe54608'
top_20 = get_top_20(user_name,url_list, model)
print(top_20)

255it [00:00, 871.20it/s]

[('https://www.corporatenepal.com/news/detail/202402', 2.4175257682800293), ('https://www.corporatenepal.com/news/detail/202280', 2.4120430946350098), ('https://www.corporatenepal.com/news/detail/202096', 2.399923324584961), ('https://www.corporatenepal.com/news/detail/202322', 2.387162446975708), ('https://www.corporatenepal.com/news/detail/201795', 2.307803153991699), ('https://www.corporatenepal.com/news/detail/202046', 2.288975715637207), ('https://www.corporatenepal.com/news/detail/201925', 2.2788162231445312), ('https://www.corporatenepal.com/news/detail/201927', 2.273982048034668), ('https://www.corporatenepal.com/news/detail/202317', 2.1893234252929688), ('https://www.corporatenepal.com/news/detail/202473', 2.186249017715454), ('https://www.corporatenepal.com/news/detail/202079', 2.1850485801696777), ('https://www.corporatenepal.com/news/detail/201844', 2.173684597015381), ('https://www.corporatenepal.com/news/detail/202248', 2.1642158031463623), ('https://www.corporatenepal.co

In [15]:
user_list = [user_name,]

In [16]:
url_to_rating = eval_sample_list(user_list,url_list)

51it [00:00, 673.99it/s]


In [17]:
url_to_rating

{'https://www.corporatenepal.com/news/detail/201726': 1.8915878534317017,
 'https://www.corporatenepal.com/news/detail/201739': 2.0202884674072266,
 'https://www.corporatenepal.com/news/detail/201741': 1.9695465564727783,
 'https://www.corporatenepal.com/news/detail/201744': 1.36106538772583,
 'https://www.corporatenepal.com/news/detail/201749': 1.5984747409820557,
 'https://www.corporatenepal.com/news/detail/201750': 1.9668601751327515,
 'https://www.corporatenepal.com/news/detail/201752': 1.701655626296997,
 'https://www.corporatenepal.com/news/detail/201755': 1.3369715213775635,
 'https://www.corporatenepal.com/news/detail/201765': 1.427971363067627,
 'https://www.corporatenepal.com/news/detail/201770': 1.9494283199310303,
 'https://www.corporatenepal.com/news/detail/201784': 1.9105578660964966,
 'https://www.corporatenepal.com/news/detail/201786': 1.4710876941680908,
 'https://www.corporatenepal.com/news/detail/201788': 1.7553730010986328,
 'https://www.corporatenepal.com/news/deta

In [72]:
top_20

[('https://www.corporatenepal.com/news/detail/202402', 2.4175257682800293),
 ('https://www.corporatenepal.com/news/detail/202280', 2.4120430946350098),
 ('https://www.corporatenepal.com/news/detail/202096', 2.399923324584961),
 ('https://www.corporatenepal.com/news/detail/202322', 2.387162446975708),
 ('https://www.corporatenepal.com/news/detail/201795', 2.307803153991699),
 ('https://www.corporatenepal.com/news/detail/202046', 2.288975715637207),
 ('https://www.corporatenepal.com/news/detail/201925', 2.2788162231445312),
 ('https://www.corporatenepal.com/news/detail/201927', 2.273982048034668),
 ('https://www.corporatenepal.com/news/detail/202317', 2.1893234252929688),
 ('https://www.corporatenepal.com/news/detail/202473', 2.186249017715454),
 ('https://www.corporatenepal.com/news/detail/202079', 2.1850485801696777),
 ('https://www.corporatenepal.com/news/detail/201844', 2.173684597015381),
 ('https://www.corporatenepal.com/news/detail/202248', 2.1642158031463623),
 ('https://www.corp

In [73]:
top_20[0][0]

'https://www.corporatenepal.com/news/detail/202402'

In [83]:
def context_giver(top_20):
    return_val = []
    for item in top_20:
        item_df = pre.df[pre.df.url == item[0]].iloc[0]
        item_val = {
            "url": item_df.url,
            "date": item_df.date.split()[0],
            "label": item_df.label,
            "all_content": item_df.all_content,
            "main_topic": item_df.main_topic,
        }
        return_val.append(item_val)
    return return_val
context_giver(top_20)        

[{'url': 'https://www.corporatenepal.com/news/detail/202402',
  'date': '2077-07-03',
  'label': 'संस्थान',
  'all_content': 'चार्जिङ स्टेसन विद्युत महशुल लागू, प्रतियुनिट रुपियाँ पैसा मात्र काठमाडौं। विद्युतीय चार्जिङ स्टेसन विद्युत् महसुल सवारी चार्ज गरेबापत महसुल लागू नेपाल विद्युत् प्राधिकरण असोज लागू हुनेगरी विद्युतीय सवारी सञ्चालन कम्पनी महसुलबारे पत्राचार विद्युत् महसुल निर्धारण आयोग महिनाअघि शुल्क प्राधिकरण कार्यान्वयन निर्धारण महसुलअनुसार विद्युतीय चार्ज चार्जिङ स्टेसन प्रतियुनिट रुपियाँ पैसा तिर् स्टेसन यसअघि प्रतियुनिट रुपियाँ तिर् चार्जिङ स्टेसन विद्युतीय सवारी चार्ज प्रतियुनिट रुपियाँ पैसा प्रतिशत शुल्क लिएर सवारी चार्ज सुविधा सरकार विद्युतीय चार्जिङ स्टेसन विद्युत् चार्ज शुल्क निर्धारण नगर् चार्जिङ स्टेसन समस्या उपत्यका चारवटा सार्वजनिक विद्युतीय सञ्चालन सुन्दर यातायात आफ्नो चार्जिङ स्टेसन महँगो शुल्क तिर्न सजिलो अध्यक्ष भेषबहादुर थापा बताए गोरखापत्र दैनिकबाट',
  'main_topic': 'शिक्षा'},
 {'url': 'https://www.corporatenepal.com/news/detail/202280',
  'date': '2077-06-28',